# Transformer model

In [1]:
import os, gc, torch
from transformers import pipeline

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DEVICE = 0
DTYPE = torch.float16

/home/cairo/code/transformers-course/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tasks

`pipeline()` function connects a model with its necessary preprocessing and postprocessing steps

In [3]:
classifier = pipeline("sentiment-analysis",
    torch_dtype=DTYPE,
    device_map="auto")
classifier("I've been waiting for a HuggingFace course my whole life")


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5914.68it/s]


[{'label': 'POSITIVE', 'score': 0.9515040516853333}]

In [4]:
classifier(["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"])

[{'label': 'POSITIVE', 'score': 0.9599125981330872},
 {'label': 'NEGATIVE', 'score': 0.99945467710495}]

The pipeline() function supports multiple modalities, allowing you to work with text, images, audio, and even multimodal tasks. 

In [ ]:
# empty cache / memory
del classifier
gc.collect()
torch.cuda.empty_cache()

**Zero-shot classification**

- classify text that haven't been labelled
- you can specify which labels to use (or not)
- you don’t need to fine-tune the model on your data to use it

**Text generation**

- the model autocomplete the remaining text
- involves randonmenss (you could not get the same results)

In [6]:
generator = pipeline("text-generation",
        torch_dtype=DTYPE,  # critical for 3060
        device_map="auto",
        max_length=30)

generator("In this course, we will teach you how to")

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 326/326 [00:00<00:00, 344.08it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'In this course, we will teach you how to use Python to analyze data from the internet. We will cover a range of topics, including web scraping'}]

In [7]:
generator = pipeline("text-generation",
        model="HuggingFaceTB/SmolLM2-360M",
        max_length=30,
        torch_dtype=DTYPE,  # critical for 3060
        device_map="auto"
)

generator("In this course, we will teach you how to")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3130.91it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'In this course, we will teach you how to do it, so you can start improving your writing today!\n\nIn this course, you will'}]

In [8]:
del generator
gc.collect()
torch.cuda.empty_cache()

**Mask filling**

- fill the blanks
- <mask> referred as a *mask token*

In [9]:
from transformers import pipeline

unmasker = pipeline("fill-mask",
    torch_dtype=DTYPE,  # critical for 3060
    device_map="auto")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 106/106 [00:00<00:00, 3598.66it/s]
[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.19677734375,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.040283203125,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

In [10]:
del unmasker
gc.collect()
torch.cuda.empty_cache()

**NER - named entity recognition**

- person
- location
- organizations

In [15]:
ner = pipeline("ner",
    torch_dtype=DTYPE,
)

ner("My name is Batman and I work at Hugging Face in Gothan.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6821.96it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity': 'I-PER',
  'score': np.float32(0.99256),
  'index': 4,
  'word': 'Batman',
  'start': 11,
  'end': 17},
 {'entity': 'I-ORG',
  'score': np.float32(0.9430584),
  'index': 9,
  'word': 'Hu',
  'start': 32,
  'end': 34},
 {'entity': 'I-ORG',
  'score': np.float32(0.8728547),
  'index': 10,
  'word': '##gging',
  'start': 34,
  'end': 39},
 {'entity': 'I-ORG',
  'score': np.float32(0.90155435),
  'index': 11,
  'word': 'Face',
  'start': 40,
  'end': 44},
 {'entity': 'I-LOC',
  'score': np.float32(0.9943614),
  'index': 13,
  'word': 'Got',
  'start': 48,
  'end': 51},
 {'entity': 'I-LOC',
  'score': np.float32(0.99338937),
  'index': 14,
  'word': '##han',
  'start': 51,
  'end': 54}]

In [16]:
del ner
gc.collect()
torch.cuda.empty_cache()

**Question asnwering**

- answers questions using information from a given context

In [17]:
from transformers.models import depth_anything
qa = pipeline("question-answering",
    dtype=DTYPE)

qa(question="Where do I work?",
   context="My name is Sylvain and I work at Hugging Face in Brooklyn",)

KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

Atualmente não existe mais este recurso, ou mudou de nome.

**Summarization**

- reducing text into a shorter text
- keep most important aspects referenced in the text

In [19]:
summarizer = pipeline("text2text-generation", model="facebook/bart-large-cnn",
        torch_dtype=DTYPE)

summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of 
    graduates in traditional engineering disciplines such as mechanical, civil, 
    electrical, chemical, and aeronautical engineering declined, but in most of 
    the premier American universities engineering curricula now concentrate on 
    and encourage largely the study of engineering science. As a result, there 
    are declining offerings in engineering subjects dealing with infrastructure, 
    the environment, and related issues, and greater concentration on high 
    technology subjects, largely supporting increasingly complex scientific 
    developments. While the latter is important, it should not be at the expense 
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other 
    industrial countries in Europe and Asia, continue to encourage and advance 
    the teaching of engineering. Both China and India, respectively, graduate 
    six and eight times as many traditional engineers as does the United States. 
    Other industrial countries at minimum maintain their output, while America 
    suffers an increasingly serious decline in the number of engineering graduates 
    and a lack of well-educated engineers.
"""
)

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

Também não está mais disponível.
Uma saída é usar Native Model & Tokenizer API

In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

text ="""
    summarize: 
    America has changed dramatically during recent years. Not only has the number of 
    graduates in traditional engineering disciplines such as mechanical, civil, 
    electrical, chemical, and aeronautical engineering declined, but in most of 
    the premier American universities engineering curricula now concentrate on 
    and encourage largely the study of engineering science. As a result, there 
    are declining offerings in engineering subjects dealing with infrastructure, 
    the environment, and related issues, and greater concentration on high 
    technology subjects, largely supporting increasingly complex scientific 
    developments. While the latter is important, it should not be at the expense 
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other 
    industrial countries in Europe and Asia, continue to encourage and advance 
    the teaching of engineering. Both China and India, respectively, graduate 
    six and eight times as many traditional engineers as does the United States. 
    Other industrial countries at minimum maintain their output, while America 
    suffers an increasingly serious decline in the number of engineering graduates 
    and a lack of well-educated engineers.
"""

inputs = tokenizer(text, return_tensors="pt", truncation=True).input_ids
outputs = model.generate(inputs, max_new_tokens=50, min_length=20, do_sample=False)

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(summary)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 4995.58it/s]


the number of graduates in traditional engineering disciplines has declined. in most of the premier american universities engineering curricula now concentrate on and encourage largely the study of engineering science. rapidly developing economies such as China and India continue to encourage


You can use the same for translation

In [25]:
# Use a standard text-generation model
translator = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct", device_map="auto")

messages = [
    {"role": "system", "content": "You are a professional translator. Translate the text to English."},
    {"role": "user", "content": "Ce cours est produit par Hugging Face."}
]

response = translator(messages, max_new_tokens=50)
print(response[0]['generated_text'][-1]['content'])

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 431.95it/s]
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


That's correct. The text "Ce cours est produit par Hugging Face" is in French, and it translates to "This course is produced by Hugging Face" in English.


In [26]:
del translator
gc.collect()
torch.cuda.empty_cache()

#### Image and Audio pipeline

In [27]:
image_classifier = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224"
)

result = image_classifier(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)

print(result)

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 11527.41it/s]


[{'label': 'lynx, catamount', 'score': 0.43339964747428894}, {'label': 'cougar, puma, catamount, mountain lion, painter, panther, Felis concolor', 'score': 0.034805234521627426}, {'label': 'snow leopard, ounce, Panthera uncia', 'score': 0.03240462765097618}, {'label': 'Egyptian cat', 'score': 0.023973938077688217}, {'label': 'tiger cat', 'score': 0.02290233038365841}]


In [28]:
transcriber = pipeline(
    task="automatic-speech-recognition", 
    model="openai/whisper-large-v3"
)

result = transcriber(
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
)

print(result)

Loading weights: 100%|██████████| 1259/1259 [00:00<00:00, 7877.78it/s]
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The cle

{'text': ' I have a dream that one day this nation will rise up and live out the true meaning of its creed.'}


In [29]:
del transcriber
gc.collect()
torch.cuda.empty_cache()

You can also combine data from multiple sources (audio, text, image) - this is a poweful application of Transformer